In [ ]:
import os
import netCDF4 as nc
import pandas as pd
import numpy as np
import numpy.ma as ma
import xarray as xr
import netCDF4 as nc
from netCDF4 import Dataset
import shapefile
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import time
import sys
import pickle

# 设置参数
OUTPUT_FOLDER = "default"
CAL_IMP_METHOD = "shap" # 计算特征重要性的方法，包括：perm、MDI、shap
feature_selection_strategy = "RFE"
SCALE_FACTOR = 0.01
NCPU = 30
SCORE="KGE"
BASIN_NAME = "CN"
# MODELS = ["XGB",  "CB", "LGBM"] #, SHAP
# MODELS = ["LGBM", "XGB", "CB", "RF"] # MDI
# ============================== ST =================================
SOIL_DEPTH = "10cm"
TRAIN_DATA_FILENAME = f"STmerge_db_daily_train_by_Date.parquet"
VALIDATION_DATA_FILENAME = f"STmerge_db_daily_valid_by_Date.parquet"
# TEST_DATA_FILENAME = f"STmerge_db_daily_test_by_Date.parquet"
TARGET = f'OBS_ST_{SOIL_DEPTH}'
# ============================== SKT =================================
SOIL_DEPTH = "0cm"
START_TEST_DATE = "2012-01-01"
TRAIN_DATA_FILENAME = f'SKTmerge_daily_10cm_1km_train_by_Date_{START_TEST_DATE}.parquet'
VALIDATION_DATA_FILENAME = f'SKTmerge_daily_10cm_1km_valid_by_Date_{START_TEST_DATE}.parquet'
TARGET = f'OBS_SKT'
# =============================================================================
# 设置路径
BASE_PATH = "/home/yfdong/data/work/STmerge/v1.0/"
DATA_PATH = "/raid61/yfdong/data/work/STmerge/v1.0"
DB_PATH = os.path.join(DATA_PATH , "dataframe/database")
SAVE_PATH = os.path.join(DATA_PATH , "dataframe/train_output", SOIL_DEPTH, OUTPUT_FOLDER)
FEATURE_PATH = os.path.join(SAVE_PATH, 'FeatureSelection')
OPTUNA_PATH = os.path.join(SAVE_PATH, "Optuna")


In [ ]:
sys.path.append(f"{BASE_PATH}/code/Library")
from MergeST import save_files, optuna_MLmodel, preprocess_features, load_data
# Load data
train_data = load_data(TRAIN_DATA_FILENAME, DB_PATH)
test_data = load_data(VALIDATION_DATA_FILENAME, DB_PATH)

In [ ]:

import optuna
import time
start = time.time()    
NTRIAL=50
# MODELS = [ "LGBM", "XGB", "CB"]
MODELS = ["CB"]
if 'RF' in MODELS:
    print('RF')
    modelNCPU=10
    optunaNCPU=10
else:
    modelNCPU=20
    optunaNCPU=20
score_df = pd.DataFrame()
MODEL_NAME = "CB" 
for MODEL_NAME in MODELS:
    print(f"#----------------{MODEL_NAME}---------------#")
    # 按照"Date"列进行排序
    # -----------------------------------特征选择---------------------------------
    features = pd.read_csv(os.path.join(FEATURE_PATH, f"{feature_selection_strategy}_{CAL_IMP_METHOD}_{MODEL_NAME}_{BASIN_NAME}_{TARGET}_subset_feature.csv"))["Feature"].tolist()
    print(features)
    # -------------------------------提取特征和目标变量-------------------------------
    X_train_scaled, y_train, scaler = preprocess_features(train_data, features, TARGET, SCALE_FACTOR)
    # ---------------------------------超参数优化-------------------------------
    study, optuna_score, DEFscore = optuna_MLmodel(X_train_scaled, y_train, SCORE, BASIN_NAME, MODEL_NAME, NTRIAL, modelNCPU,optunaNCPU)
    template_score_df_list = {
            "BASIN":BASIN_NAME,
            "MODEL":MODEL_NAME,
            "Optuna Score":optuna_score,
            "Def Score":DEFscore
    }
    print(template_score_df_list)
    template_score_df = pd.DataFrame([template_score_df_list])
    score_df = pd.concat([score_df, template_score_df], ignore_index=True)
    # -----------------------------保存history-----------------------------------
    # ...（其余代码略）
    data = []
    for trial in study.trials:
        data.append([trial.number, trial.value])
    optuna_history_df = pd.DataFrame(data, columns=['Iteration', f'Objective Score: {SCORE}'])
    # 将DataFrame存储为CSV文件
    save_files(OPTUNA_PATH, f'{BASIN_NAME}_{MODEL_NAME}_{TARGET}_{feature_selection_strategy}_{CAL_IMP_METHOD}_optimization_history_{SCORE}.csv', optuna_history_df)    
    # -----------------------------保存study---------------------------------------
    study_file = os.path.join(OPTUNA_PATH , f'{BASIN_NAME}_{MODEL_NAME}_{TARGET}_{feature_selection_strategy}_{CAL_IMP_METHOD}_study.pkl')
    import pickle
    with open(study_file, 'wb') as f:
        pickle.dump(study, f)
save_files(OPTUNA_PATH, f"CN_{MODEL_NAME}_{TARGET}_optuna_metric.csv", score_df)   
end = time.time()
print(f"Elapse Time: {end - start}Seconds")

